In [1]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)

# 12 完整建模工作流演示

本 Notebook 演示完整的风控建模流程，包括：
1. 数据加载与探索
2. 特征工程
3. 特征筛选
4. 模型训练
5. 模型评估
6. 生成模型报告

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

from hscredit import init_setting
from hscredit.core import metrics, viz

init_setting()

# 加载数据
_roots = [Path.cwd(), Path.cwd() / 'examples', Path.cwd().parent]
_fp = None
for _r in _roots:
    for _n in ('hscredit_yyp.xlsx', 'hengshucredit_yyp.xlsx'):
        if (_r / _n).is_file():
            _fp = _r / _n
            break
    if _fp is not None:
        break
if _fp is None:
    raise FileNotFoundError('请将 hscredit_yyp.xlsx 放在 examples/')

df = pd.read_excel(_fp)

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

print("=" * 50)
print("数据加载完成")
print("=" * 50)
print(f"样本数: {len(df):,}")
print(f"特征数: {df.shape[1]}")
print(f"放款时间范围: {df['放款时间'].min()} ~ {df['放款时间'].max()}")
print(f"坏样本率: {df['target'].mean():.2%}")
print("=" * 50)

FileNotFoundError: 请将 hscredit_yyp.xlsx 放在 examples/

## 第一步: 数据探索

In [ ]:
# 选择建模特征
numeric_features = [
    '中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3',
    '轻花老客海纳子分V1', '天创小额网贷分', '衡枢鉴真分老客版'
]

# 移除缺失值过多的特征
df_model = df[numeric_features + ['target']].copy()
# df_model = df_model.dropna()

print(f"建模样本数: {len(df_model):,}")
print(f"建模特征数: {len(numeric_features)}")
print(f"\n特征描述统计:")
display(df_model[numeric_features].describe())

建模样本数: 970
建模特征数: 8

特征描述统计:


,中智小牛分C3,珊瑚92,极光欺诈分6v1,青云24,占信V3,轻花老客海纳子分V1,天创小额网贷分,衡枢鉴真分老客版
count,307.0000,264.0000,308.0000,970.0000,965.0000,938.0000,970.0000,970.0000
mean,635.9316,624.8636,0.3079,604.3258,573.6383,0.0653,710.3392,0.0946
std,91.2117,70.3443,0.2424,64.9345,62.6627,0.0566,52.5425,0.0524
min,464.0000,440.0000,0.0025,372.0000,349.0000,0.0090,497.0000,0.0095
25%,564.5000,589.5000,0.1028,561.2500,534.0000,0.0271,674.5000,0.0541
50%,629.0000,617.0000,0.2375,603.0000,578.0000,0.0455,712.0000,0.0838
75%,704.0000,653.0000,0.4688,647.0000,616.0000,0.0817,747.0000,0.1239
max,850.0000,850.0000,0.8808,850.0000,762.0000,0.3300,888.0000,0.3076


## 第二步: 数据划分

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model[numeric_features]
y = df_model['target']

# 划分训练集、测试集、OOT集
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_test, X_oot, y_test, y_oot = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("数据集划分:")
print(f"  训练集: {len(X_train):,} ({len(X_train)/len(X):.1%})")
print(f"  测试集: {len(X_test):,} ({len(X_test)/len(X):.1%})")
print(f"  OOT集:  {len(X_oot):,} ({len(X_oot)/len(X):.1%})")
print(f"\n各集坏账率:")
print(f"  训练集: {y_train.mean():.2%}")
print(f"  测试集: {y_test.mean():.2%}")
print(f"  OOT集:  {y_oot.mean():.2%}")

数据集划分:
  训练集: 582 (60.0%)
  测试集: 194 (20.0%)
  OOT集:  194 (20.0%)

各集坏账率:
  训练集: 16.67%
  测试集: 17.01%
  OOT集:  16.49%


## 第三步: 特征筛选

In [ ]:
from hscredit.core.selectors import IVSelector, CorrSelector, CompositeFeatureSelector

# IV筛选
iv_selector = IVSelector(threshold=0.02)
iv_selector.fit(X_train, y_train)

print("IV筛选结果:")
iv_df = pd.DataFrame({
    '特征': iv_selector.feature_names_in_,
    'IV': iv_selector.scores_
}).sort_values('IV', ascending=False)
display(iv_df)

# 保留高IV特征
high_iv_features = iv_df[iv_df['IV'] >= 0.02]['特征'].tolist()
print(f"\n保留特征 ({len(high_iv_features)}个): {high_iv_features}")

IV筛选结果:


,特征,IV
青云24,青云24,1.4767
占信V3,占信V3,1.4113
天创小额网贷分,天创小额网贷分,1.2424
珊瑚92,珊瑚92,1.1955
中智小牛分C3,中智小牛分C3,1.0899
极光欺诈分6v1,极光欺诈分6v1,1.0694
轻花老客海纳子分V1,轻花老客海纳子分V1,0.0020
衡枢鉴真分老客版,衡枢鉴真分老客版,0.0000



保留特征 (6个): ['青云24', '占信V3', '天创小额网贷分', '珊瑚92', '中智小牛分C3', '极光欺诈分6v1']


In [ ]:
# 相关性筛选（基于IV，保留IV较高的特征）
corr_selector = CorrSelector(threshold=0.95, target='target')
corr_selector.fit(X_train[high_iv_features], y_train)

selected_features = corr_selector.selected_features_
print(f"相关性筛选后保留特征 ({len(selected_features)}个): {selected_features}")
if len(corr_selector.dropped_) > 0:
    print("\n剔除特征:")
    display(corr_selector.dropped_)

相关性筛选后保留特征 (6个): ['占信V3', '极光欺诈分6v1', '中智小牛分C3', '青云24', '珊瑚92', '天创小额网贷分']


### 评分卡模型训练

In [ ]:
from hscredit import OptimalBinning, ScoreCard, LogisticRegression


binner = OptimalBinning(target='target', max_bins=5, min_bin_size=0.01, monotonic_trend='auto_asc_desc', missing_separate=True)
binner.fit(df_model[selected_features + ['target']])

lr = LogisticRegression(target='target', C=1, max_iter=64)
lr.fit(binner.transform(df_model[selected_features + ['target']], metric='woe'))

scorecard = ScoreCard(binner=binner, lr_model=lr, target='target', base_score=600, pdo=50, step=50)
scorecard.fit(df_model[selected_features + ['target']], input_type='raw')

ScoreCard(base_score=600, binner=OptimalBinning(),
          lr_model=LogisticRegression(C=1, max_iter=64, target='target'),
          pdo=50, step=50)

In [ ]:
lr.summary()

,Coef.,Std.Err,z,P>|z|,[0.025,0.975],VIF
const,-1.5970,0.0920,-17.3525,0.0000,-1.7774,-1.4166,1.0625
占信V3,0.7066,0.2478,2.8520,0.0043,0.2210,1.1922,1.0033
极光欺诈分6v1,0.5599,0.3819,1.4660,0.1426,-0.1887,1.3085,1.0917
中智小牛分C3,0.6699,0.3475,1.9276,0.0539,-0.0113,1.3510,1.0444
青云24,0.7605,0.2201,3.4560,0.0005,0.3292,1.1918,1.0041
珊瑚92,0.5291,0.4197,1.2607,0.2074,-0.2935,1.3518,1.1068
天创小额网贷分,0.7275,0.3523,2.0652,0.0389,0.0371,1.4179,1.0131


In [ ]:
from hscredit.core.viz import plot_weights

plot_weights(lr, figsize=(8, 4));

NameError: name 'cp_model' is not defined

In [ ]:
df_model.loc[:0, selected_features]

,占信V3,极光欺诈分6v1,中智小牛分C3,青云24,珊瑚92,天创小额网贷分
0,NaN,NaN,NaN,656,NaN,630


In [ ]:
scorecard.get_detailed_score(df_model.loc[:0])

样本信息                    占信V3                                 极光欺诈分6v1                         ...   青云24 珊瑚92                                  天创小额网贷分                                                                                                 评分分析
  样本索引        总分     截距分数  原始值             分箱    WOE        分数      原始值              分箱    WOE  ...     分数  原始值             分箱      WOE      分数      原始值             分箱 WOE       分数                                                                     评分原因
0    0 2127.5200 971.6600  NaN  [-inf, 460.4) 0.0561 1173.7900      NaN  [-inf, 0.2374) 0.0705  ... 7.0000  NaN  [-inf, 567.5) -23.0283 -2.6900 630.0000  [-inf, 687.0) NaN -17.1900  占信V3提升1179.0分(当前1173.8分); 天创小额网贷分拉低27.1分(当前-17.2分); 珊瑚92提升0.0分(当前-2.7分)

[1 rows x 28 columns]

In [ ]:
binner.get_bin_table('占信V3')

,分箱,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,0,"[-inf, 460.4)",49,33,16,0.0505,0.0408,0.0988,0.3265,0.8830,0.0511,0.2939,1.9552,0.0508,1.9552,0.0508,33,16,0.0579
1,1,"[460.4, 563.0)",331,269,62,0.3412,0.3329,0.3827,0.1873,0.1394,0.0069,0.2939,1.1216,0.0630,1.2290,0.1475,302,78,0.1077
2,2,"[563.0, 608.0)",286,234,52,0.2948,0.2896,0.3210,0.1818,0.1029,0.0032,0.2939,1.0887,0.0371,1.1688,0.3697,536,130,0.1391
3,3,"[608.0, 671.0)",245,216,29,0.2526,0.2673,0.1790,0.1184,-0.4010,0.0354,0.2939,0.7087,-0.0984,1.0450,0.6955,752,159,0.0508
4,4,"[671.0, +inf)",54,51,3,0.0557,0.0631,0.0185,0.0556,-1.2262,0.0547,0.2939,0.3326,-0.0393,1.0052,1.0000,803,162,0.0062
5,-1,missing,5,5,0,0.0052,0.0062,0.0000,0.0000,-23.0283,0.1425,0.2939,0.0000,-0.0052,1.0000,0.0000,808,162,0.0000


In [ ]:
scorecard.scorecard_points()

,变量名称,变量含义,变量分箱,对应分数,WOE值
0,基础分,截距项（基准分数）,-,971.6600,NaN
1,占信V3,,"[-inf, 460.4)",-45.0100,0.8830
2,占信V3,,"[460.4, 563.0)",-7.1000,0.1394
3,占信V3,,"[563.0, 608.0)",-5.2400,0.1029
4,占信V3,,"[608.0, 671.0)",20.4400,-0.4010
5,占信V3,,"[671.0, +inf)",62.5000,-1.2262
6,占信V3,,缺失值,1173.7900,-23.0283
7,极光欺诈分6v1,,"[-inf, 0.2374)",22.0000,-0.5448
8,极光欺诈分6v1,,"[0.2374, 0.321)",19.0800,-0.4725
9,极光欺诈分6v1,,"[0.321, 0.697)",-5.6800,0.1406


In [ ]:
scorecard.score_odds_reference

,评分,理论Odds,好客户:坏客户,理论逾期率,理论逾期率(%),对数Odds
0,350,1120.0000,1120.0:1,0.9991,99.9108%,7.0211
1,400,560.0000,560.0:1,0.9982,99.8217%,6.3279
2,450,280.0000,280.0:1,0.9964,99.6441%,5.6348
3,500,140.0000,140.0:1,0.9929,99.2908%,4.9416
4,550,70.0000,70.0:1,0.9859,98.5915%,4.2485
5,600,35.0000,35.0:1,0.9722,97.2222%,3.5553
6,650,17.5000,17.5:1,0.9459,94.5946%,2.8622
7,700,8.7500,8.8:1,0.8974,89.7436%,2.1691
8,750,4.3750,4.4:1,0.8140,81.3953%,1.4759
9,800,2.1875,2.2:1,0.6863,68.6275%,0.7828


In [ ]:
scorecard.lr_model.summary()

,Coef.,Std.Err,z,P>|z|,[0.025,0.975],VIF
const,-1.5970,0.0920,-17.3525,0.0000,-1.7774,-1.4166,1.0625
占信V3,0.7066,0.2478,2.8520,0.0043,0.2210,1.1922,1.0033
极光欺诈分6v1,0.5599,0.3819,1.4660,0.1426,-0.1887,1.3085,1.0917
中智小牛分C3,0.6699,0.3475,1.9276,0.0539,-0.0113,1.3510,1.0444
青云24,0.7605,0.2201,3.4560,0.0005,0.3292,1.1918,1.0041
珊瑚92,0.5291,0.4197,1.2607,0.2074,-0.2935,1.3518,1.1068
天创小额网贷分,0.7275,0.3523,2.0652,0.0389,0.0371,1.4179,1.0131


In [ ]:
scorecard.score_to_bad_rate_table(scorecard.predict(df_model[selected_features]), df_model['target'])

,评分区间,样本数,坏样本数,坏样本率,好样本数,Odds,累计好样本占比,累计坏样本占比,KS
0,"(857.45, 922.635]",113,37,32.74%,76,2.05,0.0941,0.2284,0.1343
1,"(922.635, 946.625]",83,22,26.51%,61,2.77,0.1696,0.3642,0.1946
2,"(946.625, 949.713]",108,19,17.59%,89,4.68,0.2797,0.4815,0.2018
3,"(949.713, 955.44]",84,13,15.48%,71,5.46,0.3676,0.5617,0.1942
4,"(955.44, 975.564]",144,32,22.22%,112,3.50,0.5062,0.7593,0.2531
5,"(975.564, 981.929]",61,11,18.03%,50,4.55,0.5681,0.8272,0.2591
6,"(981.929, 1001.249]",110,16,14.55%,94,5.88,0.6844,0.9259,0.2415
7,"(1001.249, 1028.05]",76,4,5.26%,72,18.00,0.7735,0.9506,0.1771
8,"(1028.05, 1068.677]",94,7,7.45%,87,12.43,0.8812,0.9938,0.1126
9,"(1068.677, 2277.067]",97,1,1.03%,96,96.00,1.0000,1.0000,0.0000


In [ ]:
scorecard.get_score_reference_by_prob(prob_range=(0, 0.99), n_points=20)

,理论逾期率,理论逾期率(%),理论Odds,评分
0,0.0001,0.0100%,0.0001,1520.8400
1,0.0522,5.2200%,0.0551,1065.5900
2,0.1043,10.4300%,0.1164,1011.5800
3,0.1564,15.6400%,0.1854,978.0300
4,0.2085,20.8500%,0.2634,952.6900
5,0.2606,26.0600%,0.3524,931.6900
6,0.3127,31.2700%,0.4550,913.2700
7,0.3648,36.4800%,0.5743,896.4700
8,0.4169,41.6900%,0.7150,880.6700
9,0.4690,46.9000%,0.8832,865.4200


In [ ]:
"""评分卡 Python 部署代码（自动生成）"""
import numpy as np
import pandas as pd


def calculate_score(row: dict) -> float:
    """计算单条样本的评分卡分数.

    :param row: 样本特征字典
    :return: 评分
    """
    score = 971.665  # base_score

    # 占信V3
    val = row.get("占信V3")
    if val < 460.4:
        score += -45.0102
    elif val >= 460.4 and val < 563.0:
        score += -7.1049
    elif val >= 563.0 and val < 608.0:
        score += -5.2444
    elif val >= 608.0 and val < 671.0:
        score += 20.4404
    elif val >= 671.0:
        score += 62.5037
    elif val is None or (isinstance(val, float) and np.isnan(val)):
        score += 1173.7875

    # 极光欺诈分6v1
    val = row.get("极光欺诈分6v1")
    if val < 0.2374:
        score += 22.0046
    elif val >= 0.2374 and val < 0.321:
        score += 19.0836
    elif val >= 0.321 and val < 0.697:
        score += -5.6801
    elif val >= 0.697:
        score += -27.169
    elif val is None or (isinstance(val, float) and np.isnan(val)):
        score += -2.3401

    # 中智小牛分C3
    val = row.get("中智小牛分C3")
    if val < 495.3:
        score += -52.9657
    elif val >= 495.3 and val < 611.7:
        score += -7.1195
    elif val >= 611.7 and val < 698.0:
        score += 7.0341
    elif val >= 698.0 and val < 793.4:
        score += 54.0034
    elif val >= 793.4:
        score += 1168.9376
    elif val is None or (isinstance(val, float) and np.isnan(val)):
        score += -2.7111

    # 青云24
    val = row.get("青云24")
    if val < 481.5:
        score += 88.4269
    elif val >= 481.5 and val < 597.0:
        score += -16.9887
    elif val >= 597.0 and val < 613.5:
        score += 15.2273
    elif val >= 613.5 and val < 623.0:
        score += 124.2124
    elif val >= 623.0:
        score += 7.0019

    # 珊瑚92
    val = row.get("珊瑚92")
    if val < 567.5:
        score += -23.4252
    elif val >= 567.5 and val < 643.5:
        score += 11.1754
    elif val >= 643.5 and val < 722.4:
        score += 18.0342
    elif val >= 722.4:
        score += 35.0688
    elif val is None or (isinstance(val, float) and np.isnan(val)):
        score += -2.6921

    # 天创小额网贷分
    val = row.get("天创小额网贷分")
    if val < 687.0:
        score += -17.1935
    elif val >= 687.0:
        score += 9.8851

    return score


def batch_calculate_score(df: pd.DataFrame) -> pd.Series:
    """批量计算评分."""
    return df.apply(lambda row: calculate_score(row.to_dict()), axis=1)

In [ ]:
print(scorecard.export_deployment_code(language='python'))

"""评分卡 Python 部署代码（自动生成）"""
import numpy as np
import pandas as pd


def calculate_score(row: dict) -> float:
    """计算单条样本的评分卡分数.

    :param row: 样本特征字典
    :return: 评分
    """
    score = 971.665  # base_score

    # 占信V3
    val = row.get("占信V3")
    if val < 460.4:
        score += -45.0102
    elif val >= 460.4 and val < 563.0:
        score += -7.1049
    elif val >= 563.0 and val < 608.0:
        score += -5.2444
    elif val >= 608.0 and val < 671.0:
        score += 20.4404
    elif val >= 671.0:
        score += 62.5037
    elif val is None or (isinstance(val, float) and np.isnan(val)):
        score += 1173.7875

    # 极光欺诈分6v1
    val = row.get("极光欺诈分6v1")
    if val < 0.2374:
        score += 22.0046
    elif val >= 0.2374 and val < 0.321:
        score += 19.0836
    elif val >= 0.321 and val < 0.697:
        score += -5.6801
    elif val >= 0.697:
        score += -27.169
    elif val is None or (isinstance(val, float) and np.isnan(val)):
        score += -2.3401

    #

In [ ]:
scorecard.export_pmml()

4月 10, 2026 12:52:17 上午 sklearn2pmml.pipeline.PMMLPipeline encodePMML
警告: Model verification data is not set. Use the 'sklearn2pmml.pipeline.PMMLPipeline.verify(X)' method to correct this deficiency
4月 10, 2026 12:52:17 上午 [com.sklearn2pmml.Main]  run
信息: Generated PMML file /Users/xiaoxi/Desktop/itlubber/model_platform_codebuddy/hscredit/examples/scorecard.pmml (13'027 bytes)

Recommended PMML deployment tools:
  * MS Excel       https://xlsboost.com
  * Java           https://github.com/jpmml/jpmml-evaluator
  * Python         https://github.com/jpmml/jpmml-evaluator-python
  * R              https://github.com/jpmml/jpmml-evaluator-r
  * Apache Spark   https://github.com/jpmml/jpmml-evaluator-spark
  * REST API       https://github.com/openscoring/openscoring

Please contact info@openscoring.io for commercial licensing options
PMML 文件已导出至: scorecard.pmml


In [ ]:
from pypmml import Model

model = Model.load('scorecard.pmml')

pd.DataFrame(zip(
    scorecard.predict(df_model),
    model.predict(df_model[lr.feature_names_in_])['predicted_score']
)).assign(diff=lambda x: x[0] - x[1]).describe()

,0,1,diff
count,970.0000,970.0000,970.0000
mean,1008.2590,1008.2590,0.0000
std,183.7190,183.7190,0.0000
min,857.4506,857.4506,-0.0000
25%,949.7132,949.7132,0.0000
50%,975.5643,975.5643,0.0000
75%,1009.4746,1009.4746,0.0000
max,2277.0666,2277.0666,0.0000


In [ ]:
pd.DataFrame(zip(scorecard.predict(df_model), batch_calculate_score(df_model))).assign(diff=lambda x: x[0] - x[1]).describe()

,0,1,diff
count,970.0000,970.0000,970.0000
mean,1008.2590,1008.2590,-0.0000
std,183.7190,183.7190,0.0000
min,857.4506,857.4507,-0.0002
25%,949.7132,949.7132,-0.0000
50%,975.5643,975.5643,-0.0000
75%,1009.4746,1009.4745,0.0000
max,2277.0666,2277.0667,0.0001


## 评分卡模型报告

基于上面训练好的 `scorecard` 生成评分卡模型报告，并分别查看训练集、测试集和 OOT 集的效果。

In [ ]:
from hscredit.report.model_report import auto_model_report, QuickModelReport

scorecard_report_file = 'scorecard_model_report.xlsx'
scorecard_report_html = 'scorecard_model_report.html'

scorecard_report = auto_model_report(
    scorecard,
    X_train[selected_features],
    y_train,
    X_test[selected_features],
    y_test,
    excel_path=scorecard_report_file,
    html_path=scorecard_report_html,
    verbose=False,
    n_bins=8,
    bin_method='quantile',
    model_name='ScoreCard评分卡模型',
)

pd.DataFrame(
    {
        '输出文件': [scorecard_report_file, scorecard_report_html],
        '说明': ['训练集/测试集评分卡模型报告（Excel）', '训练集/测试集评分卡模型报告（HTML）'],
    }
)

,输出文件,说明
0,scorecard_model_report.xlsx,训练集/测试集评分卡模型报告（Excel）
1,scorecard_model_report.html,训练集/测试集评分卡模型报告（HTML）


In [ ]:
display(scorecard_report.get_metrics())
display(scorecard_report.get_feature_importance(top_n=10))
display(scorecard_report.get_bin_table('train', method='quantile', max_n_bins=8))
display(scorecard_report.get_bin_table('test', method='quantile', max_n_bins=8))

,统计项,训练集,测试集
0,KS,0.3072,0.1971
1,AUC,0.7002,0.6114
2,样本数,582,194.0000
3,坏样本率,0.1667,0.1701
4,PSI,\,0.0069


,特征重要性,IV,KS,PSI
青云24,0.1924,0.0504,0.1464,0.0531
天创小额网贷分,0.1840,0.0437,0.1196,0.0209
占信V3,0.1787,0.2664,0.2124,0.0010
中智小牛分C3,0.1694,1.7436,0.0639,0.0281
极光欺诈分6v1,0.1416,0.4742,0.1010,0.1502
珊瑚92,0.1338,4.9695,0.0495,0.0133


,指标名称,指标含义,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,__score__,模型评分,"[-inf, 935.7984)",102,69,33,0.1753,0.1423,0.3402,0.3235,0.8718,0.1726,5.1264,1.9412,0.2000,1.9412,0.2000,69,33,0.1979
1,__score__,模型评分,"[935.7984, 951.5737)",106,84,22,0.1821,0.1732,0.2268,0.2075,0.2697,0.0145,5.1264,1.2453,0.0546,1.5865,0.3262,153,55,0.2515
2,__score__,模型评分,"[951.5737, 974.1704)",86,69,17,0.1478,0.1423,0.1753,0.1977,0.2085,0.0069,5.1264,1.1860,0.0323,1.4694,0.4792,222,72,0.2845
3,__score__,模型评分,"[974.1704, 981.6907)",73,60,13,0.1254,0.1237,0.1340,0.1781,0.0800,0.0008,5.1264,1.0685,0.0098,1.3896,0.6651,282,85,0.2948
4,__score__,模型评分,"[981.6907, 1043.3123)",124,112,12,0.2131,0.2309,0.1237,0.0968,-0.6242,0.0669,5.1264,0.5806,-0.1135,1.1853,1.0000,394,97,0.1876
5,__score__,模型评分,"[1043.3123, +inf)",91,91,0,0.1564,0.1876,0.0000,0.0000,-25.9273,4.8647,5.1264,0.0000,-0.1853,1.0000,0.0000,485,97,0.0000
6,__score__,模型评分,合计,582,485,97,1.0000,1.0000,1.0000,0.1667,0.0000,0.0000,5.1264,1.0000,1.0000,1.0000,1.0000,1605,439,0.2948


,指标名称,指标含义,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,__score__,模型评分,"[-inf, 916.7467)",9,4,5,0.0464,0.0248,0.1515,0.5556,1.8080,0.2290,1.8336,3.2660,0.1102,3.2660,0.1102,4,5,0.1267
1,__score__,模型评分,"[916.7467, 948.4856)",26,20,6,0.1340,0.1242,0.1818,0.2308,0.3809,0.0219,1.8336,1.3566,0.0552,1.8476,0.1866,24,11,0.1843
2,__score__,模型评分,"[948.4856, 955.8322)",39,34,5,0.2010,0.2112,0.1515,0.1282,-0.3320,0.0198,1.8336,0.7537,-0.0620,1.2711,0.1672,58,16,0.1246
3,__score__,模型评分,"[955.8322, 973.7037)",9,9,0,0.0464,0.0559,0.0000,0.0000,-23.6382,1.3214,1.8336,0.0000,-0.0486,1.1333,0.0996,67,16,0.0687
4,__score__,模型评分,"[973.7037, 975.5643)",9,7,2,0.0464,0.0435,0.0606,0.2222,0.3321,0.0057,1.8336,1.3064,0.0149,1.1502,0.1355,74,18,0.0858
5,__score__,模型评分,"[975.5643, 977.2584)",12,8,4,0.0619,0.0497,0.1212,0.3333,0.8918,0.0638,1.8336,1.9596,0.0633,1.2436,0.2815,82,22,0.1573
6,__score__,模型评分,"[977.2584, 1007.9742)",37,30,7,0.1907,0.1863,0.2121,0.1892,0.1296,0.0033,1.8336,1.1122,0.0264,1.2091,0.5563,112,29,0.1831
7,__score__,模型评分,"[1007.9742, +inf)",53,49,4,0.2732,0.3043,0.1212,0.0755,-0.9206,0.1686,1.8336,0.4437,-0.2091,1.0000,0.0000,161,33,0.0000
8,__score__,模型评分,合计,194,161,33,1.0000,1.0000,1.0000,0.1701,0.0000,0.0000,1.8336,1.0000,1.0000,1.0000,1.0000,582,150,0.1843


In [ ]:
scorecard_oot_report = QuickModelReport(
    scorecard,
    X_train[selected_features],
    y_train,
    X_oot[selected_features],
    y_oot,
)

# 报告中 test 数据集即 OOT
display(scorecard_oot_report.get_metrics())
display(scorecard_oot_report.get_bin_table('test', method='quantile', max_n_bins=8))

# 可以用 add_dataset 添加更多数据集
# scorecard_oot_report.add_dataset("oot2", "OOT2", X_oot2, y_oot2)

,统计项,训练集,测试集
0,KS,0.3072,0.3696
1,AUC,0.7002,0.6956
2,样本数,582,194.0000
3,坏样本率,0.1667,0.1649
4,PSI,\,0.0414


,指标名称,指标含义,分箱标签,样本总数,好样本数,坏样本数,样本占比,好样本占比,坏样本占比,坏样本率,分档WOE值,分档IV值,指标IV值,LIFT值,坏账改善,累积LIFT值,累积坏账改善,累积好样本数,累积坏样本数,分档KS值
0,__score__,模型评分,"[-inf, 922.6345)",14,9,5,0.0722,0.0556,0.1562,0.3571,1.0341,0.1041,5.9222,2.1652,0.0906,2.1652,0.0906,9,5,0.1007
1,__score__,模型评分,"[922.6345, 946.6251)",24,19,5,0.1237,0.1173,0.1562,0.2083,0.2869,0.0112,5.9222,1.2630,0.0371,1.5954,0.1450,28,10,0.1397
2,__score__,模型评分,"[946.6251, 975.5643)",44,33,11,0.2268,0.2037,0.3438,0.2500,0.5232,0.0733,5.9222,1.5156,0.1512,1.5526,0.4046,61,21,0.2797
3,__score__,模型评分,"[975.5643, 1001.249)",37,29,8,0.1907,0.1790,0.2500,0.2162,0.3340,0.0237,5.9222,1.3108,0.0732,1.4774,0.7575,90,29,0.3507
4,__score__,模型评分,"[1001.249, 1009.3712)",19,18,1,0.0979,0.1111,0.0312,0.0526,-1.2685,0.1013,5.9222,0.3191,-0.0739,1.3179,0.7835,108,30,0.2708
5,__score__,模型评分,"[1009.3712, 1019.1356)",9,9,0,0.0464,0.0556,0.0000,0.0000,-23.6012,1.3112,5.9222,0.0000,-0.0486,1.2372,0.7420,117,30,0.2153
6,__score__,模型评分,"[1019.1356, 1051.4911)",19,17,2,0.0979,0.1049,0.0625,0.1053,-0.5182,0.0220,5.9222,0.6382,-0.0393,1.1687,1.0000,134,32,0.1728
7,__score__,模型评分,"[1051.4911, +inf)",28,28,0,0.1443,0.1728,0.0000,0.0000,-24.7362,4.2754,5.9222,0.0000,-0.1687,1.0000,0.0000,162,32,0.0000
8,__score__,模型评分,合计,194,162,32,1.0000,1.0000,1.0000,0.1649,0.0000,0.0000,5.9222,1.0000,1.0000,1.0000,1.0000,709,189,0.3507
